# Reto 03 — Q-Learning: exploración vs. explotación

**Curso:** Aprendizaje por Refuerzo — Universidad EIA  
**Integrantes:** Nombre Apellido  
**Correos:** correo@eia.edu.co  
**Fecha:** AAAA-MM-DD

> **Pregunta guía:** ¿cómo cambia el aprendizaje de una política cuando Q-Learning usa ε-greedy, ε-greedy con inicialización optimista, UCB o Softmax en entornos con distinta estructura de recompensas?

## Objetivos

- Implementar Q-Learning tabular respetando la API de Gymnasium.
- Separar la actualización de valor de la estrategia de selección.
- Comparar cuatro mecanismos de exploración bajo condiciones reproducibles.
- Evaluar aprendizaje, estabilidad y desempeño de la política final.
- Sustentar decisiones con métricas, gráficas y evidencia experimental.


## Organización en dos clases

### Clase 1 — Construcción y verificación

1. Reconocer `FrozenLake-v1`, `Blackjack-v1` y `Taxi-v4`.
2. Implementar las cuatro estrategias de selección.
3. Implementar el ciclo de Q-Learning y sus pruebas unitarias pequeñas.
4. Ejecutar un experimento piloto en FrozenLake.

**Hito al terminar:** las funciones pasan las verificaciones y una corrida produce curvas coherentes.

### Clase 2 — Experimento y comunicación

1. Ejecutar la comparación principal en los tres escenarios.
2. Evaluar las políticas sin exploración.
3. Realizar sensibilidad de hiperparámetros.
4. Interpretar resultados, documentar IA y redactar conclusiones.

**Hito al terminar:** tabla resumen, visualizaciones, respuestas y conclusiones completas.


## 1. Preparación del entorno


In [ ]:
from __future__ import annotations

from collections.abc import Callable

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 2026
ALPHA = 0.10
GAMMA = 0.99
N_RUNS = 30

ENV_CONFIG = {
    "FrozenLake-v1": {"episodes": 2_000, "max_steps": 100, "kwargs": {"is_slippery": True}},
    "Blackjack-v1": {"episodes": 10_000, "max_steps": 100, "kwargs": {"sab": True}},
    "Taxi-v4": {"episodes": 2_000, "max_steps": 200, "kwargs": {}},
}

STRATEGIES = {
    "epsilon_greedy": {"epsilon": 0.10, "q0": 0.0},
    "optimistic": {"epsilon": 0.01, "q0": 5.0},
    "ucb": {"c": 1.0, "q0": 0.0},
    "softmax": {"temperature": 0.50, "q0": 0.0},
}


## 2. Reconocimiento de los escenarios

Para cada entorno, reporte número de estados y acciones, rango de recompensas, condición terminal y dificultad esperada. Observe que `reset()` devuelve `(observation, info)` y `step()` devuelve `(observation, reward, terminated, truncated, info)`. Un episodio termina si `terminated or truncated`. Blackjack devuelve una tupla, por lo que se suministra una codificación tabular reproducible.


In [ ]:
def tabular_space_size(observation_space) -> int:
    """Número de estados de un espacio Discrete o Tuple de Discrete."""
    if isinstance(observation_space, gym.spaces.Discrete):
        return int(observation_space.n)
    if isinstance(observation_space, gym.spaces.Tuple):
        return int(np.prod([space.n for space in observation_space.spaces]))
    raise TypeError("Este reto solo admite observaciones discretas.")


def encode_observation(observation, observation_space) -> int:
    """Convierte una observación discreta, incluida una tupla, en un índice."""
    if isinstance(observation_space, gym.spaces.Discrete):
        return int(observation)
    dimensions = tuple(space.n for space in observation_space.spaces)
    return int(np.ravel_multi_index(tuple(observation), dimensions))


for env_id, config in ENV_CONFIG.items():
    env = gym.make(env_id, **config["kwargs"])
    observation, info = env.reset(seed=SEED)
    print(env_id, "| estados tabulares:", tabular_space_size(env.observation_space), "| acciones:", env.action_space.n)
    print("  observación inicial:", observation, "| acciones válidas:", list(range(env.action_space.n)))
    env.close()


### Análisis previo

Complete antes de entrenar:

| Entorno | Estados | Acciones | Recompensas relevantes | ¿Qué dificulta explorar? |
|---|---:|---:|---|---|
| FrozenLake-v1 | TODO | TODO | TODO | TODO |
| Blackjack-v1 | TODO | TODO | TODO | TODO |
| Taxi-v4 | TODO | TODO | TODO | TODO |


## 3. Estrategias de exploración

Todas las funciones deben devolver una acción válida y resolver empates aleatoriamente. `counts[s, a]` registra cuántas veces se eligió la acción `a` en el estado `s`.

- **ε-greedy:** acción aleatoria con probabilidad ε; en otro caso, una acción greedy.
- **ε-greedy optimista:** misma regla, pero la tabla inicia en `Q0 > 0`; explique por qué esto incentiva explorar.
- **UCB:** seleccione primero acciones no visitadas; luego maximice `Q(s,a) + c sqrt(log(N(s)+1) / N(s,a))`.
- **Softmax:** muestree según `softmax(Q(s,·)/τ)` usando estabilización numérica.


In [ ]:
def random_argmax(values: np.ndarray, rng: np.random.Generator) -> int:
    """Devuelve aleatoriamente uno de los índices con valor máximo."""
    # TODO
    raise NotImplementedError


def select_action(
    q: np.ndarray,
    counts: np.ndarray,
    state: int,
    strategy: str,
    rng: np.random.Generator,
    *,
    epsilon: float = 0.10,
    c: float = 1.0,
    temperature: float = 0.50,
) -> int:
    """Selecciona una acción usando la estrategia solicitada."""
    # TODO: validar strategy, epsilon, c y temperature.
    # TODO: implementar epsilon_greedy, optimistic, ucb y softmax.
    raise NotImplementedError


### Pruebas pequeñas de selección


In [ ]:
# Ejecute esta celda después de implementar select_action.
q_test = np.array([[1.0, 3.0, 3.0, 0.0]])
n_test = np.array([[5, 5, 5, 0]])

# TODO: compruebe acciones válidas, empates no deterministas, UCB con acciones
# no visitadas, reproducibilidad y errores ante hiperparámetros inválidos.


## 4. Q-Learning tabular

Implemente la actualización:

$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha [R_{t+1} + \gamma(1-d)\max_a Q(S_{t+1},a)-Q(S_t,A_t)],$$

donde `d` es verdadero cuando la transición termina el MDP (`terminated`). Para un truncamiento por límite de tiempo, documente si realiza *bootstrap* y mantenga la decisión consistente. No actualice hacia el valor del estado terminal.


In [ ]:
def train_q_learning(
    env_id: str,
    strategy: str,
    episodes: int,
    max_steps: int,
    alpha: float,
    gamma: float,
    seed: int,
    env_kwargs: dict | None = None,
    **strategy_params,
) -> tuple[np.ndarray, pd.DataFrame, np.ndarray]:
    """Entrena Q-Learning y devuelve Q, historial por episodio y conteos."""
    # TODO: validar entradas y crear env/rng con semillas controladas.
    # TODO: usar encode_observation y tabular_space_size.
    # TODO: inicializar Q con q0 y counts con ceros.
    # TODO: guardar episode, return, length, success y td_error_mean.
    # TODO: cerrar el entorno aun si ocurre un error.
    raise NotImplementedError


### Contrato mínimo y verificaciones

Para una corrida corta compruebe:

- `Q.shape == (n_states, n_actions)` y todos sus valores son finitos.
- Hay exactamente una fila por episodio.
- `1 <= length <= max_steps`.
- `counts.sum()` coincide con el total de pasos registrados.
- La misma semilla reproduce resultados; otra semilla puede cambiarlos.
- Con `gamma=0`, el objetivo no contiene el valor del siguiente estado.
- Una transición terminal usa como objetivo únicamente la recompensa.


In [ ]:
# TODO: entrene 20 episodios de FrozenLake y escriba asserts para el contrato.


## 5. Evaluación sin exploración

Evaluar con la misma exploración usada durante el entrenamiento mezcla dos efectos. Implemente una evaluación separada que seleccione acciones greedy, no actualice `Q` y use semillas distintas a las de entrenamiento. En empates, conserve la selección aleatoria.


In [ ]:
def evaluate_policy(
    env_id: str,
    q: np.ndarray,
    n_episodes: int,
    max_steps: int,
    seed: int,
    env_kwargs: dict | None = None,
) -> pd.DataFrame:
    """Evalúa la política greedy sin modificar Q."""
    # TODO: devolver episode, return, length y success.
    raise NotImplementedError


## 6. Experimento piloto — cierre de la clase 1

Entrene las cuatro estrategias en `FrozenLake-v1` durante 300 episodios y una sola corrida. Grafique retorno y éxito suavizados con una media móvil de 50 episodios. Use este piloto para detectar errores; **no** lo presente como evidencia concluyente.


In [ ]:
# TODO: ejecutar el piloto, consolidar historiales y graficar.


## 7. Comparación principal — clase 2

Use los valores de `ENV_CONFIG`, al menos 30 corridas independientes y las mismas semillas por entorno para las cuatro estrategias. Esto empareja la aleatoriedad experimental sin reutilizar el mismo generador. Evalúe cada tabla final durante 100 episodios con semillas nuevas.

Registre por episodio: entorno, algoritmo, corrida, episodio, retorno, longitud, éxito y error TD medio. Registre por evaluación: retorno, longitud y éxito. Justifique cualquier reducción de corridas realizada por límites computacionales.


In [ ]:
def run_experiment(
    env_config: dict,
    strategies: dict,
    n_runs: int = N_RUNS,
    base_seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Ejecuta entrenamiento y evaluación para todos los escenarios."""
    # TODO: use seed = base_seed + run para entrenamiento.
    # TODO: use una familia de semillas distinta para evaluación.
    raise NotImplementedError


# training_results, evaluation_results = run_experiment(ENV_CONFIG, STRATEGIES)


## 8. Métricas y visualizaciones obligatorias

Construya una tabla por entorno y algoritmo con:

1. Retorno medio de entrenamiento en el último 10 % de episodios.
2. Área bajo la curva del retorno medio (calidad durante el aprendizaje).
3. Tasa de éxito final durante evaluación.
4. Retorno medio de evaluación.
5. Longitud media de evaluación.
6. Intervalo de confianza del 95 % entre corridas (indique el método).

Incluya, como mínimo:

- Curvas de retorno por episodio, media móvil y banda de IC 95 %.
- Curvas de éxito acumulado o móvil.
- Comparación del retorno de evaluación por entorno.
- Mapa de calor de `max_a Q(s,a)` o visualización de política para al menos un entorno.

No promedie directamente episodios de entornos con escalas de recompensa distintas en una única cifra global.


In [ ]:
# TODO: tabla resumen con incertidumbre.
# TODO: cuatro visualizaciones completas (título, ejes, unidades y leyenda).


## 9. Sensibilidad de hiperparámetros

Seleccione el entorno donde las estrategias difirieron más y compare:

| Estrategia | Valores mínimos |
|---|---|
| ε-greedy | ε ∈ {0.01, 0.10, 0.30} |
| Optimista | Q0 ∈ {1, 5, 10}, con ε = 0.01 |
| UCB | c ∈ {0.25, 1, 2} |
| Softmax | τ ∈ {0.10, 0.50, 1.0} |

Puede usar 10 corridas, pero mantenga constantes episodios, α, γ y semillas. Presente una gráfica o tabla y explique el patrón; no elija un valor mirando únicamente una corrida.


In [ ]:
# TODO: ejecutar y analizar la sensibilidad.


## 10. Interpretación

Responda citando valores, tablas o figuras del notebook:

1. ¿Qué estrategia aprendió más rápido en cada entorno?
2. ¿La estrategia con mejor curva de entrenamiento produjo también la mejor política greedy?
3. ¿Dónde ayudó y dónde perjudicó la inicialización optimista? Relaciónelo con recompensas negativas y estados poco visitados.
4. ¿Cómo se comportó UCB cuando muchos pares estado–acción fueron visitados pocas veces?
5. ¿Qué efecto tuvo la temperatura de Softmax sobre estabilidad y desempeño?
6. ¿Qué estrategia fue más sensible a la estocasticidad de FrozenLake?
7. En Blackjack, ¿qué diferencias aparecen entre manos con y sin as utilizable?
8. ¿Existe una estrategia ganadora para los tres escenarios y todas las métricas?
9. Si el costo de interacción fuera alto, ¿qué estrategia recomendaría y por qué?
10. Mencione dos amenazas a la validez del experimento.


## 11. Uso de inteligencia artificial

**Herramienta(s):**  
**Prompts principales:**  
1. TODO  
2. TODO  

**Sugerencias aceptadas o rechazadas y por qué:** TODO  
**Cambios manuales:** TODO  
**Cómo se verificó el código y la interpretación:** TODO


## 12. Conclusiones

Redacte entre cuatro y seis conclusiones sustentadas en resultados. Evite repetir definiciones o afirmar causalidad sin evidencia.

1. TODO
2. TODO
3. TODO
4. TODO


## Lista de verificación de entrega

- [ ] Implementé y validé las cuatro estrategias.
- [ ] Q-Learning trata correctamente estados terminales y truncamientos.
- [ ] Comparé tres entornos con semillas y presupuesto equivalentes.
- [ ] Separé entrenamiento y evaluación greedy.
- [ ] Reporté variabilidad entre corridas.
- [ ] Incluí tabla, cuatro visualizaciones y sensibilidad.
- [ ] Respondí todas las preguntas con evidencia.
- [ ] Documenté el uso de IA y escribí conclusiones.
- [ ] Reinicié el kernel y ejecuté todas las celdas.
- [ ] Exporté y revisé el HTML.
